# Model comparison — cross-check and independent evaluation (Colab, T4)

Runs **the committed script** from the project repository with the same seed,
dataset and flags as the run on the deployment machine, and additionally scores
every trained model on BAFMD before its session is cleared.

Three purposes:

1. **Verify the accuracies.** The comparison is seeded, so a second machine should
   land close. That is a cross-machine reproducibility check.
2. **Give a clean hardware comparison.** Only the hardware differs this time, so
   the latency gap is not confounded with dataset and timing protocol as it was
   in the earlier Colab run.
3. **Make the comparison discriminating.** On curated data the ranking rests on
   between three and fifteen errors out of 818 — too few to separate the middle
   of the field. On BAFMD the errors run to hundreds, so differences are real.

The models are never saved to disk, so the BAFMD evaluation has to happen inside
this run. Doing it later would mean retraining all five.

**The dissertation reports the deployment-machine figures.** Latency must be
measured on the hardware the system runs on. This notebook corroborates the
accuracies and adds the independent evaluation.

## Before you start

On your Mac, from the repository root:

    git add -A && git commit -m "Add extra-eval to model comparison" && git push
    zip -qr ~/Downloads/m_dataset.zip m/dataset
    zip -qr ~/Downloads/bafmd_crops.zip data/bafmd_crops

Upload **both** zips to the top level of your Google Drive.
Then **Runtime → Change runtime type → T4 GPU → Save.**

The push matters: this notebook clones from GitHub, so an uncommitted script
would not be the one that runs.

### 1. Confirm the GPU

In [ ]:
import tensorflow as tf
print('TensorFlow:', tf.__version__)
print('GPU:', tf.config.list_physical_devices('GPU'))
!nvidia-smi -L

### 2. Mount Drive and unpack both datasets

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/ds
!unzip -q '/content/drive/MyDrive/m_dataset.zip'   -d /content/ds
!unzip -q '/content/drive/MyDrive/bafmd_crops.zip' -d /content/ds

import os
for label, base in [('m/dataset', '/content/ds/m/dataset'),
                    ('bafmd_crops', '/content/ds/data/bafmd_crops')]:
    print(label)
    for c in ['with_mask', 'without_mask']:
        print('   ', c, len(os.listdir(f'{base}/{c}')))

print('\nExpect  m/dataset: 2165 / 1930     bafmd_crops: 1354 / 328')
print('If these differ, stop — the run would not be comparable.')

### 3. Clone the repository
Cloning rather than pasting code guarantees this is the committed script, and doubles as a check that the repository runs from a clean environment.

In [ ]:
!git clone -q https://github.com/Sreeshanth19/msc-surveillance /content/msc-surveillance
!cd /content/msc-surveillance && git log --oneline -1
!pip install -q tf_keras
print('\nRecord that commit hash — it belongs in the provenance appendix.')

### 4. Run

Identical flags to the deployment-machine run — 12 epochs, batch 32, seed 42, 0.2
validation split, median of 20 timed batches after 3 warm-ups — plus the BAFMD
evaluation. Roughly 20–45 minutes on a T4.

In [ ]:
!cd /content/msc-surveillance && python -m scripts.compare_models \
    --dataset /content/ds/m/dataset \
    --extra-eval bafmd=/content/ds/data/bafmd_crops \
    --epochs 12 --seed 42 \
    --out /content/out 2>&1 | tee /content/run_log_colab.txt

### 5. The tables

In [ ]:
print(open('/content/msc-surveillance/out/comparison_table.txt').read())

### 6. Download

Save these into the repository as
`results/model_comparison/colab_t4_crosscheck/`, together with a note recording
the commit hash, the TensorFlow version and the GPU model from step 1.

In [ ]:
from google.colab import files
for f in ['comparison_table.txt', 'results.json', 'comparison_accuracy.png']:
    files.download(f'/content/msc-surveillance/out/{f}')
files.download('/content/run_log_colab.txt')

### How to read the result

#### The in-distribution table

Accuracies should land close to the deployment-machine run — inceptionv3 0.9963,
resnet50 0.9927, efficientnetb0 0.9927, vgg16 0.9890, mobilenetv2 0.9817. They
will not match to four decimal places: seeding fixes the data order and
initialisation, but GPU kernels are not bit-identical to CPU ones and the
TensorFlow version differs. **Values within a few tenths of a percent, in a
similar order, is a pass** — it shows the comparison is reproducible rather than
an artefact of one machine.

Remember what those figures rest on: 3, 6, 6, 9 and 15 errors out of 818. The
extremes are separable; the middle three are not. Do not rank them in the
dissertation as though they were.

#### Latency

It will drop sharply and the spread should compress. On the Mac CPU the range was
3.0–56.3 ms, a 19× spread. A GPU parallelises the heavy convolutions, so VGG16's
disadvantage shrinks. **That compression is the finding** — the same lesson as
the MPS-versus-CPU detector result: the backend changes the ranking, so a model
chosen on one device is not automatically right for another.

#### The BAFMD table — the interesting one

Watch two columns rather than accuracy, which is misleading at roughly 80/20 class
balance: **`mask rec`** (recall on masked faces) and **`nomask prec`** (precision
on the non-compliance flag). The deployed classifier scores 0.7725 and 0.5008 —
it wrongly flags about one masked person in four, and half of everything it calls
non-compliant is wrong.

Three outcomes, all publishable:

- **All five cluster low.** The failure lies in the training distribution, not the
  architecture. This is the strongest result: no backbone choice would have
  rescued it, and the remedy is training data, not model selection.
- **One or two are markedly better.** Architecture choice matters on hard data
  even though it is invisible on curated data, and MobileNetV2 was a costlier
  choice than the in-distribution table suggested.
- **All five score far above 0.80.** Then the gap is a training-regime effect
  rather than an architectural one — these models were trained here on
  `m/dataset` with frozen backbones, whereas the deployed model came from the
  baseline author. Report it as such and do not claim it contradicts the deployed
  result.

Whichever occurs, the honest limit is unchanged: BAFMD crops come from social
media, so demographic representation, image domain, crop provenance and mask-type
diversity are confounded. The claim is that performance degrades on independent,
diverse data — not that demographics alone caused it.